# 01 – Exploratory Data Analysis (EDA)

This notebook explores the ACIS auto insurance dataset. It focuses on:

- Understanding the structure and quality of the data
- Creating basic risk metrics (loss ratio, margin, claim flag)
- Visualising distributions and key risk drivers


In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.load_data import load_raw_data
from src.features.build_features import add_core_features

pd.set_option("display.max_columns", 100)
sns.set(style="whitegrid")

## 1. Load the raw dataset

In [ ]:
DATA_PATH = os.path.join("data", "raw", "insurance.csv")

# This will raise FileNotFoundError if the file does not exist.
df = load_raw_data(DATA_PATH)

df.head()

## 2. Basic structure and summary statistics

In [ ]:
df.info()

In [ ]:
df.describe(include="all").transpose().head(20)

## 3. Add core risk features

In [ ]:
df = add_core_features(df)
df[["TotalPremium", "TotalClaims", "LossRatio", "Margin", "HasClaim"]].head()

## 4. Missing values overview

In [ ]:
missing = df.isna().mean().sort_values(ascending=False)
missing[missing > 0].head(20)

In [ ]:
plt.figure(figsize=(8, 4))
missing[missing > 0].plot(kind="bar")
plt.title("Missing value ratio by column")
plt.ylabel("Fraction of missing values")
plt.tight_layout()
plt.show()

## 5. Overall portfolio metrics

In [ ]:
if {"TotalPremium", "TotalClaims"}.issubset(df.columns):
    total_premium = df["TotalPremium"].sum()
    total_claims = df["TotalClaims"].sum()
    overall_loss_ratio = total_claims / total_premium if total_premium > 0 else np.nan
    print(f"Total premium: {total_premium:,.0f}")
    print(f"Total claims:  {total_claims:,.0f}")
    print(f"Overall loss ratio: {overall_loss_ratio:.3f}")

## 6. Loss ratio by province, vehicle type, and gender

In [ ]:
group_cols = ["Province", "VehicleType", "Gender"]

for col in group_cols:
    if col in df.columns:
        summary = (
            df.groupby(col)
            .agg(
                policies=("PolicyID", "nunique") if "PolicyID" in df.columns else ("HasClaim", "count"),
                total_premium=("TotalPremium", "sum"),
                total_claims=("TotalClaims", "sum"),
                loss_ratio=("LossRatio", "mean"),
                claim_frequency=("HasClaim", "mean"),
            )
            .sort_values("loss_ratio", ascending=False)
        )
        print(f"\n=== {col} ===")
        display(summary.head(10))

## 7. Time trends over the 18-month period

In [ ]:
if "TransactionMonth" in df.columns:
    monthly = (
        df.groupby("TransactionMonth")
        .agg(
            total_premium=("TotalPremium", "sum"),
            total_claims=("TotalClaims", "sum"),
            claim_frequency=("HasClaim", "mean"),
        )
        .sort_index()
    )
    monthly["loss_ratio"] = monthly["total_claims"] / monthly["total_premium"]

    fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
    monthly[["total_premium", "total_claims"]].plot(ax=axes[0])
    axes[0].set_title("Monthly total premium vs claims")

    monthly[["claim_frequency", "loss_ratio"]].plot(ax=axes[1])
    axes[1].set_title("Monthly claim frequency & loss ratio")

    plt.tight_layout()
    plt.show()
else:
    print("Column 'TransactionMonth' not found in dataset.")

## 8. Example creative plot – Loss ratio by province

In [ ]:
if "Province" in df.columns and "LossRatio" in df.columns:
    prov = (
        df.groupby("Province")
        .agg(
            loss_ratio=("LossRatio", "mean"),
            claim_frequency=("HasClaim", "mean"),
            policies=("PolicyID", "nunique") if "PolicyID" in df.columns else ("HasClaim", "count"),
        )
        .sort_values("loss_ratio", ascending=False)
    )

    plt.figure(figsize=(10, 5))
    sns.barplot(x=prov.index, y=prov["loss_ratio"])
    plt.xticks(rotation=45)
    plt.ylabel("Average loss ratio")
    plt.title("Average loss ratio by province")
    plt.tight_layout()
    plt.show()
else:
    print("Required columns not found to plot loss ratio by province.")

## 9. Notes for your report

Use the outputs and plots above to answer questions like:

- Which provinces or vehicle types are the riskiest / safest?
- How does loss ratio vary by gender?
- Are there clear time trends in claim frequency or severity?

You can write your narrative in `reports/interim_report.md` and later
in the final report template.
